### Update `openalex.works.work_authors` — Author ID Matching & Minting

Assigns `author_id` to work_authors records where `author_id IS NULL`. Resolves institution IDs from `work_authors.raw_affiliation_strings` (written by `UpdateWorkAuthors`). Does NOT update affiliations.

In [0]:
DECLARE OR REPLACE VARIABLE max_updated_date TIMESTAMP DEFAULT to_timestamp('1900-01-01');
SET VARIABLE max_updated_date = COALESCE((SELECT MAX(updated_at) - INTERVAL 1 SECOND FROM openalex.works.work_authors), to_timestamp('1900-01-01'));
-- SET VARIABLE max_updated_date = to_timestamp('2025-12-20');
SELECT max_updated_date;

-- oxjob #649 rematch-on-authorship-change. Mode & seat cap come from author_rematch_control
-- (single row; empty table = defaults below). mode: 'log_only' (write author_rematch_log, NO
-- nulls, zero effect on matching) | 'off' | 'on' (apply: worklist + null/insert/delete + rematch).
CREATE TABLE IF NOT EXISTS openalex.authors.author_rematch_control (mode STRING, seat_cap INT);

DECLARE OR REPLACE VARIABLE rematch_mode STRING DEFAULT 'log_only';

SET VARIABLE rematch_mode = COALESCE((SELECT ANY_VALUE(mode) FROM openalex.authors.author_rematch_control), 'log_only');

DECLARE OR REPLACE VARIABLE rematch_seat_cap INT DEFAULT 250000;

SET VARIABLE rematch_seat_cap = COALESCE((SELECT ANY_VALUE(seat_cap) FROM openalex.authors.author_rematch_control), 250000);

CREATE TABLE IF NOT EXISTS openalex.authors.author_rematch_worklist (
    work_id BIGINT, change_classes ARRAY<STRING>, eligible BOOLEAN, est_seats INT, admitted_at TIMESTAMP);

CREATE TABLE IF NOT EXISTS openalex.authors.author_rematch_applied (
    run_date DATE, work_id BIGINT, change_classes ARRAY<STRING>, eligible BOOLEAN, est_seats INT, admitted_at TIMESTAMP);

CREATE TABLE IF NOT EXISTS openalex.authors.author_rematch_pins (
    work_id BIGINT, author_sequence BIGINT, author_id BIGINT, reason STRING, pinned_at TIMESTAMP);


### Rematch-on-authorship-change detection — oxjob #649 (LOG ONLY)

Snapshot `openalex.authors.author_rematch_log`, rebuilt each run (`rematch_mode='log_only'`: **no nulls, no worklist, zero effect on matching**). Two classes over ALL work ids, tagged `eligible`:
- `foreign_family`: bound `author_id` is a **different surname family** from the current `works_base` name — the strict 'very wrong / different person' bar (wave-S foreign-family predicate: surname not shared/contained/1-edit/swapped; display-word intersection empty; #608 judged-same overlays; junk/markup + CJK/Cyrillic/Greek/Arabic abstention). Opus precision ~87% (vs 35% for bare names_compatible). `name_diverged` = current name != stored work_authors name (change-driven subset).
- `size_grew`/`size_shrank`: `works_base` author count != `work_authors` seat count.

Holds: `orcid_anchor`, `curated`. Action IS wired (next cell: worklist + null/insert/delete + batch-gate union) but fires ONLY when `rematch_mode='on'` — under `log_only` the worklist rebuilds EMPTY and every write is a no-op. Gate design RESOLVED 2026-07-22: old works re-enter matching via `author_rematch_worklist` ONLY (blanket id/date gate stays); worklisted records match AND mint normally. Comma-form surname-first names are reordered ('Oliveira, C.' -> 'C. Oliveira') before the display-word intersection (closes the main FP class, ~87%->~95% precision).

In [ ]:
CREATE OR REPLACE TABLE openalex.authors.author_rematch_log AS
WITH base AS (
    SELECT id AS work_id, SIZE(authorships) AS base_n, authorships,
           (id > 7000000000 AND created_date >= to_timestamp('2025-12-20')) AS eligible
    FROM openalex.works.openalex_works_base
    WHERE authorships IS NOT NULL AND SIZE(authorships) > 0 AND rematch_mode <> 'off'
),
seats AS (
    SELECT b.work_id, b.eligible, t.pos AS author_sequence,
           TRIM(t.a.raw_author_name) AS base_name, NULLIF(TRIM(t.a.raw_orcid), '') AS raw_orcid
    FROM base b LATERAL VIEW posexplode(b.authorships) t AS pos, a
),
bound AS (
    SELECT s.work_id, s.eligible, s.author_sequence, s.base_name, s.raw_orcid,
           TRIM(wa.raw_author_name) AS stored_name, wa.author_id
    FROM seats s JOIN openalex.works.work_authors wa
      ON s.work_id = wa.work_id AND s.author_sequence = wa.author_sequence
    WHERE wa.author_id IS NOT NULL
),
prof AS (SELECT id, COALESCE(NULLIF(TRIM(display_name), ''), TRIM(full_name)) AS fn, NULLIF(TRIM(orcid), '') AS orcid
         FROM openalex.authors.openalex_authors),
k AS (
    SELECT b.work_id, b.eligible, b.author_sequence, b.base_name, b.stored_name, b.author_id, b.raw_orcid,
           p.fn AS bound_name, p.orcid AS bound_orcid, NOT (b.base_name <=> b.stored_name) AS name_diverged,
           an_r.match_last AS r_last, an_r.match_first AS r_first, an_p.match_last AS p_last, an_p.match_first AS p_first
    FROM bound b JOIN prof p ON b.author_id = p.id
    JOIN openalex.authors.author_names an_r ON b.base_name = an_r.raw_author_name
    JOIN openalex.authors.author_names an_p ON p.fn = an_p.raw_author_name
    WHERE an_r.match_last IS NOT NULL AND an_p.match_last IS NOT NULL
      -- junk / placeholder / markup names excluded
      AND b.base_name NOT RLIKE '(?i)^(n/?a|null|none|unknown|anonymous|et al\.?|grd)\\b'
      AND NOT (b.base_name RLIKE '<[a-zA-Z/][^>]*>' OR b.base_name RLIKE '(?i)(vcard|begin:|;type=)')
      -- script abstention: CJK + Cyrillic/Greek + Arabic (frozen-parser low confidence)
      AND NOT (b.base_name RLIKE '[Ͱ-ϿЀ-ӿ؀-ۿᄀ-ᇿ぀-ヿ㄰-㆏㐀-䶿一-鿿가-힯豈-﫿]'
               OR p.fn RLIKE '[Ͱ-ϿЀ-ӿ؀-ۿᄀ-ᇿ぀-ヿ㄰-㆏㐀-䶿一-鿿가-힯豈-﫿]')
),
ff AS (   -- FOREIGN FAMILY = 'very wrong' (different surname family), not merely incompatible
    SELECT * FROM k
    WHERE r_last <> p_last AND LENGTH(p_last) >= 2 AND LENGTH(r_last) >= 2
      AND INSTR(p_last, r_last) = 0 AND INSTR(r_last, p_last) = 0
      AND r_last <> COALESCE(p_first, '') AND COALESCE(r_first, '') <> p_last
      -- married-name exemption: full given name agrees -> assume surname change, leave alone
      AND NOT (r_first IS NOT NULL AND r_first = p_first AND LENGTH(r_first) >= 3)
      AND SIZE(ARRAY_INTERSECT(
        FILTER(SLICE(SPLIT(TRIM(REGEXP_REPLACE(TRANSLATE(LOWER(REGEXP_REPLACE(REGEXP_REPLACE(base_name,'^([^,]+),(.*)$','$2 $1'),'[.,\\-]+',' ')),'áàäâãåéèëêíìïîóòöôõøúùüûñçýćčšžł','aaaaaaeeeeiiiioooooouuuuncyccszl'),'\\s+',' ')),' '),2,20), t->LENGTH(t)>=3),
        FILTER(SLICE(SPLIT(TRIM(REGEXP_REPLACE(TRANSLATE(LOWER(REGEXP_REPLACE(REGEXP_REPLACE(bound_name,'^([^,]+),(.*)$','$2 $1'),'[.,\\-]+',' ')),'áàäâãåéèëêíìïîóòöôõøúùüûñçýćčšžł','aaaaaaeeeeiiiioooooouuuuncyccszl'),'\\s+',' ')),' '),2,20), t->LENGTH(t)>=3)
      )) = 0
      AND levenshtein(r_last, p_last) > 1
      AND NOT (levenshtein(r_last, p_last) = 2 AND LEFT(COALESCE(r_first,''),1) = LEFT(COALESCE(p_first,''),1))
      AND NOT EXISTS (SELECT 1 FROM openalex.authors.oxjob608_namepair_j_gemini j WHERE j.full_name=bound_name AND j.raw_name=base_name AND j.judge_json:same_person='true')
      AND NOT EXISTS (SELECT 1 FROM openalex.authors.oxjob608_namepair_j_rules jr WHERE jr.full_name=bound_name AND jr.raw_name=base_name AND jr.same_person)
      AND NOT EXISTS (SELECT 1 FROM openalex.authors.oxjob608_namepair_j_opus jo WHERE jo.full_name=bound_name AND jo.raw_name=base_name AND jo.judge_json:same_person='true')
),
wa_ct AS (SELECT work_id, COUNT(*) AS wa_n FROM openalex.works.work_authors GROUP BY work_id),
size_chg AS (
    SELECT b.work_id, b.eligible, b.base_n, COALESCE(w.wa_n, 0) AS wa_n
    FROM base b LEFT JOIN wa_ct w ON b.work_id = w.work_id WHERE b.base_n <> COALESCE(w.wa_n, 0)
)
SELECT current_date() AS run_date, work_id, author_sequence,
       base_name AS raw_author_name, stored_name, author_id AS prev_author_id, bound_name,
       name_diverged, 'foreign_family' AS change_class,
       CAST(NULL AS INT) AS base_n, CAST(NULL AS INT) AS wa_n, eligible, 'dryrun' AS action,
       CASE WHEN raw_orcid IS NOT NULL AND raw_orcid = bound_orcid THEN 'orcid_anchor'
            WHEN EXISTS (SELECT 1 FROM openalex.works.work_author_claim_curations cc
                         WHERE cc.work_id = ff.work_id AND (TRIM(cc.raw_author_name) = ff.base_name OR cc.author_id = ff.author_id))
                 THEN 'curated'
            WHEN EXISTS (SELECT 1 FROM openalex.authors.author_rematch_pins pin
                         WHERE pin.work_id = ff.work_id AND pin.author_sequence = ff.author_sequence AND pin.author_id = ff.author_id)
                 THEN 'pinned' ELSE NULL END AS hold_reason,
       current_timestamp() AS detected_at
FROM ff
UNION ALL
SELECT current_date(), work_id, CAST(NULL AS BIGINT), CAST(NULL AS STRING), CAST(NULL AS STRING),
       CAST(NULL AS BIGINT), CAST(NULL AS STRING), CAST(NULL AS BOOLEAN),
       CASE WHEN base_n > wa_n THEN 'size_grew' ELSE 'size_shrank' END,
       base_n, wa_n, eligible, 'dryrun', CAST(NULL AS STRING), current_timestamp()
FROM size_chg

### Rematch action — oxjob #649 (gated: fires only when `rematch_mode='on'`)

Worklist-scoped gate relax per resolved design (2026-07-22): old works re-enter matching ONLY via `author_rematch_worklist`; admitted records match AND mint normally. Admission is capped by `rematch_seat_cap` (SEATS, not works: est_seats = ff-nulled + grew-inserts + pre-existing in-range nulls), priority foreign_family > size_grew > size_shrank, eligible first. Writes are surgical: DELETE shrank orphan seats (seq >= base_n, curation-guarded), INSERT missing grew seats (author_id NULL), NULL foreign_family seats (unheld, still bound to the logged author). `author_rematch_applied` keeps the append-only history (debounce/metrics). Standing drain at 250K seats/run is ~20 nights.

In [ ]:
CREATE OR REPLACE TABLE openalex.authors.author_rematch_worklist AS
WITH cand AS (
    SELECT work_id, ANY_VALUE(eligible) AS eligible,
           COLLECT_SET(change_class) AS change_classes,
           SUM(CASE WHEN change_class = 'foreign_family' AND hold_reason IS NULL THEN 1 ELSE 0 END) AS ff_seats,
           MAX(CASE WHEN change_class = 'size_grew' THEN base_n - wa_n ELSE 0 END) AS grew_seats,
           MAX(CASE WHEN change_class <> 'foreign_family' THEN base_n END) AS size_base_n
    FROM openalex.authors.author_rematch_log
    WHERE rematch_mode = 'on'
      AND NOT EXISTS (SELECT 1 FROM openalex.authors.author_rematch_applied ap
                      WHERE ap.work_id = author_rematch_log.work_id
                        AND ap.run_date >= DATE_SUB(current_date(), 7))
    GROUP BY work_id
    HAVING SUM(CASE WHEN change_class = 'foreign_family' AND hold_reason IS NULL THEN 1 ELSE 0 END) > 0
        OR ARRAY_CONTAINS(COLLECT_SET(change_class), 'size_grew')
        OR ARRAY_CONTAINS(COLLECT_SET(change_class), 'size_shrank')
),
nulls AS (
    SELECT c.work_id, COUNT(*) AS null_seats
    FROM cand c JOIN openalex.works.work_authors wa ON wa.work_id = c.work_id
    WHERE wa.author_id IS NULL AND (c.size_base_n IS NULL OR wa.author_sequence < c.size_base_n)
    GROUP BY c.work_id
),
ranked AS (
    SELECT c.work_id, c.change_classes, c.eligible,
           CAST(c.ff_seats + c.grew_seats + COALESCE(n.null_seats, 0) AS INT) AS est_seats,
           SUM(c.ff_seats + c.grew_seats + COALESCE(n.null_seats, 0)) OVER (
               ORDER BY CASE WHEN ARRAY_CONTAINS(c.change_classes, 'foreign_family') THEN 0
                             WHEN ARRAY_CONTAINS(c.change_classes, 'size_grew') THEN 1 ELSE 2 END,
                        c.eligible DESC, c.work_id
               ROWS UNBOUNDED PRECEDING) AS cum_seats
    FROM cand c LEFT JOIN nulls n ON c.work_id = n.work_id
)
SELECT work_id, change_classes, eligible, est_seats, current_timestamp() AS admitted_at
FROM ranked WHERE cum_seats <= rematch_seat_cap;

INSERT INTO openalex.authors.author_rematch_applied
SELECT current_date(), work_id, change_classes, eligible, est_seats, admitted_at
FROM openalex.authors.author_rematch_worklist;

DELETE FROM openalex.works.work_authors AS wa
WHERE EXISTS (
        SELECT 1 FROM openalex.authors.author_rematch_worklist wl
        JOIN openalex.authors.author_rematch_log l
          ON l.work_id = wl.work_id AND l.change_class = 'size_shrank'
        WHERE wl.work_id = wa.work_id AND wa.author_sequence >= l.base_n)
  AND NOT EXISTS (
        SELECT 1 FROM openalex.works.work_author_claim_curations cc
        WHERE cc.work_id = wa.work_id
          AND (cc.author_id = wa.author_id OR TRIM(cc.raw_author_name) = TRIM(wa.raw_author_name)))
  AND NOT EXISTS (
        SELECT 1 FROM openalex.authors.author_rematch_pins pin
        WHERE pin.work_id = wa.work_id AND pin.author_sequence = wa.author_sequence
          AND pin.author_id = wa.author_id);

INSERT INTO openalex.works.work_authors
    (work_id, author_sequence, author_id, raw_author_name, raw_affiliation_strings, is_corresponding, created_at, updated_at)
SELECT b.id, t.pos, CAST(NULL AS BIGINT), TRIM(t.a.raw_author_name), t.a.raw_affiliation_strings,
       t.a.is_corresponding, current_timestamp(), current_timestamp()
FROM openalex.works.openalex_works_base b
JOIN openalex.authors.author_rematch_worklist wl
  ON b.id = wl.work_id AND ARRAY_CONTAINS(wl.change_classes, 'size_grew')
LATERAL VIEW POSEXPLODE(b.authorships) t AS pos, a
WHERE NOT EXISTS (
    SELECT 1 FROM openalex.works.work_authors wa
    WHERE wa.work_id = b.id AND wa.author_sequence = t.pos);

MERGE INTO openalex.works.work_authors AS wa
USING (
    SELECT l.work_id, l.author_sequence, l.prev_author_id
    FROM openalex.authors.author_rematch_log l
    JOIN openalex.authors.author_rematch_worklist wl ON l.work_id = wl.work_id
    WHERE l.change_class = 'foreign_family' AND l.hold_reason IS NULL
) s
ON wa.work_id = s.work_id AND wa.author_sequence = s.author_sequence AND wa.author_id = s.prev_author_id
WHEN MATCHED THEN UPDATE SET wa.author_id = NULL, wa.updated_at = current_timestamp();

### Step 1: Get updated works that need matching

In [0]:
-- STEP 1: Create Staging Table — only records that need author matching
-- Resolves institution IDs from work_authors.raw_affiliation_strings (written by affiliations step)
CREATE OR REPLACE TABLE openalex.authors.author_matching_batch AS
WITH raw_exploded AS (
    SELECT 
        id AS work_id,
        created_date,
        POSEXPLODE(authorships) AS (author_sequence, authorship)
    FROM openalex.works.openalex_works_base
    WHERE (updated_date > max_updated_date
           OR id IN (SELECT work_id FROM openalex.authors.author_rematch_worklist))
      AND authorships IS NOT NULL 
      AND SIZE(authorships) > 0
),
-- Only keep authorships that need matching:
-- 1. No author_id assigned yet
-- 2. Meets the ID and date cutoffs for new-era matching
needs_matching AS (
    SELECT r.work_id, r.author_sequence, r.authorship,
           r.authorship.raw_author_name AS raw_author_name,
           wa.raw_affiliation_strings
    FROM raw_exploded r
    INNER JOIN openalex.works.work_authors wa
        ON r.work_id = wa.work_id AND r.author_sequence = wa.author_sequence
    WHERE wa.author_id IS NULL
      AND ((r.work_id > 7000000000 AND r.created_date >= to_timestamp('2025-12-20'))
           OR r.work_id IN (SELECT work_id FROM openalex.authors.author_rematch_worklist))
),

-- Resolve institution IDs from work_authors.raw_affiliation_strings
exploded_ras AS (
    SELECT nm.work_id, nm.author_sequence, nm.authorship, nm.raw_author_name,
           t.raw_affiliation_string
    FROM needs_matching nm
    LATERAL VIEW OUTER EXPLODE(nm.raw_affiliation_strings) t AS raw_affiliation_string
),
resolved_direct_ids AS (
    SELECT 
        e.work_id, e.author_sequence, e.authorship, e.raw_author_name,
        CASE 
            WHEN e.raw_affiliation_string IS NULL THEN NULL
            WHEN asl.institution_ids IS NOT NULL AND SIZE(asl.institution_ids) > 0 
                THEN asl.institution_ids
            ELSE NULL
        END AS direct_ids
    FROM exploded_ras e
    LEFT JOIN openalex.institutions.raw_affiliation_strings_institutions_mv asl
        ON e.raw_affiliation_string = asl.raw_affiliation_string
),

-- Expand Lineage (FOR MATCHING ONLY)
expanded_for_matching AS (
    SELECT 
        r.work_id,
        r.author_sequence,
        ARRAY_DISTINCT(FLATTEN(COLLECT_LIST(
            FLATTEN(ARRAY(
                FILTER(ARRAY(r.inst_id_scalar), x -> x IS NOT NULL),
                COALESCE(anc.ancestors, ARRAY())
            ))
        ))) as matching_institution_ids
    FROM (
        SELECT work_id, author_sequence, EXPLODE_OUTER(direct_ids) as inst_id_scalar
        FROM resolved_direct_ids
    ) r
    LEFT JOIN (
        SELECT institution_id, lineage_ids as ancestors
        FROM openalex.institutions.institution_ancestors
    ) anc ON CAST(r.inst_id_scalar AS BIGINT) = anc.institution_id
    GROUP BY r.work_id, r.author_sequence
)

SELECT 
    nm.work_id,
    nm.author_sequence,
    COALESCE(efm.matching_institution_ids, ARRAY()) as all_institution_ids,
    nm.authorship as authorship_struct,
    nm.raw_author_name
FROM needs_matching nm
LEFT JOIN expanded_for_matching efm
    ON nm.work_id = efm.work_id AND nm.author_sequence = efm.author_sequence;

### Step 2: Run Matching Algorithm Over Updated Works with ID over 7000000000

In [ ]:
CREATE OR REPLACE TABLE openalex.authors.pending_author_assignments AS
WITH 
-- 1. ENRICH BATCH DATA
-- Add Signals (Topics, Sources) and parsed name columns to batch data
enriched_batch AS (
  SELECT
    b.work_id,
    b.author_sequence,
    b.raw_author_name,
    b.all_institution_ids,
    TRANSFORM(b.all_institution_ids, x -> CONCAT('https://openalex.org/I', CAST(x AS STRING))) AS institution_ids,
    
    pn.parsed_name.first AS pn_first,
    SUBSTRING(pn.parsed_name.first, 1, 1) AS pn_first_initial,
    pn.parsed_name.middle AS pn_middle,
    pn.parsed_name.last AS pn_last,
    pn.alt_parses AS alt_parses,   -- oxjob #1341
    
    COALESCE(wtf.topics, ARRAY()) AS topics,
    
    ARRAY_DISTINCT(
      TRANSFORM(
        FILTER(w.locations, x -> x.source.id IS NOT NULL),
        x -> x.source.id
      )
    ) AS work_source_ids,
    b.authorship_struct.raw_orcid AS incoming_orcid,
    -- oxjob #1360: seat of a work the #649 worklist re-admitted tonight (log origin 'rematch')
    (rw.work_id IS NOT NULL) AS is_rematch
    
  FROM openalex.authors.author_matching_batch b
  LEFT JOIN (SELECT DISTINCT work_id FROM openalex.authors.author_rematch_worklist) rw
    ON b.work_id = rw.work_id
  LEFT JOIN openalex.authors.author_names pn 
    ON TRIM(b.raw_author_name) = pn.raw_author_name
  LEFT JOIN openalex.works.work_topics wtf 
    ON b.work_id = wtf.work_id
  -- works_base, NOT openalex_works: openalex_works is rebuilt AFTER this task
  -- in end2end, so joining it misses every new work (work_source_ids empty).
  LEFT JOIN openalex.works.openalex_works_base w 
    ON b.work_id = w.id
),

-- 2. PREPARE MATCHING INPUTS
-- Calculate Block Keys and ID arrays
authors_prepared AS (
  SELECT
    work_id,
    author_sequence,
    raw_author_name,
    pn_first,
    pn_first_initial,
    pn_middle,
    pn_last,
    alt_parses,
    -- Block Key Generation (parsed_name fields are already normalized)
    CASE
      WHEN pn_last IS NULL THEN NULL
      WHEN pn_first_initial IS NULL OR pn_first_initial = '' THEN pn_last
      ELSE CONCAT(pn_first_initial, ' ', pn_last)
    END AS block_key,
    institution_ids,
    -- Extract Topic IDs
    TRANSFORM(topics, t -> t.id) AS topic_ids,
    work_source_ids,
    usable_orcid,
    -- Seats carrying the same name and signals get the same decision; only one representative
    -- per tuple goes through the block join (a 20K-seat org roster otherwise lands on one task).
    -- is_rematch splits rematch seats into their own tuples: they are held to a different variant bar.
    is_rematch,
    xxhash64(raw_author_name, institution_ids, TRANSFORM(topics, t -> t.id), work_source_ids, usable_orcid, is_rematch) AS tuple_key
  FROM (
    SELECT *,
      -- publishers sometimes stamp one author's ORCID on every authorship of a work;
      -- an ORCID on >1 authorship of the same work is untrustworthy (name cascade decides)
      CASE WHEN COUNT(*) OVER (PARTITION BY work_id, incoming_orcid) = 1 THEN incoming_orcid END AS usable_orcid
    FROM enriched_batch
    WHERE raw_author_name IS NOT NULL
  )
),

signal_reps AS (
  SELECT * EXCEPT (rn) FROM (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY tuple_key ORDER BY work_id, author_sequence) AS rn
    FROM authors_prepared
  ) WHERE rn = 1
),

-- 2b. ORCID MATCHING (global — no block constraint)
-- Match incoming ORCID to any profile holding it. If several profiles share
-- the ORCID (splinters), take the most-cited, then most works, then oldest id.
orcid_matches AS (
  SELECT
    ap.work_id,
    ap.author_sequence,
    COUNT(DISTINCT a.author_id) AS orcid_match_count,
    MAX_BY(a.author_id, STRUCT(COALESCE(a.cited_by_count, 0), COALESCE(a.works_count, 0), -a.author_id)) AS orcid_author_id
  FROM signal_reps ap
  -- any trusted ORCID the profile has ever carried resolves to it, not just the primary (#1267);
  -- exploded so the join stays an equi-join
  JOIN (
    SELECT author_id, cited_by_count, works_count, EXPLODE(observed_orcids) AS orcid
    FROM openalex.authors.authors_for_matching
    WHERE observed_orcids IS NOT NULL AND SIZE(observed_orcids) > 0
    UNION ALL
    -- Profiles not yet in authors_for_matching: minted since its 12:21 UTC rebuild (this run can
    -- finish before or after that), or zero-works / unparseable-name. Read the mint table directly
    -- so a seat carrying an ORCID we already minted cannot mint a twin (oxjob #444; the stale-lookup
    -- nights of 2026-07-15/16 minted 59K same-ORCID profiles). Zero stats: they lose every tiebreak
    -- to an established holder. authors.orcid is the mint-time value, so honour curated removals.
    SELECT a.id AS author_id, 0 AS cited_by_count, 0 AS works_count, a.orcid
    FROM openalex.authors.authors a
    LEFT ANTI JOIN openalex.authors.authors_for_matching f ON f.author_id = a.id
    LEFT JOIN openalex.authors.author_orcid_curations oc ON oc.author_id = a.id
    WHERE a.orcid IS NOT NULL AND a.orcid <> '' AND NOT (a.orcid <=> oc.removed_orcid)
  ) a
    ON a.orcid = ap.usable_orcid
  GROUP BY ap.work_id, ap.author_sequence
),

-- 3. CANDIDATE BLOCKING
-- 3a. Primary block (unchanged): the seat's block key equals the profile's block key.
primary_candidates AS (
  SELECT
    e.tuple_key,
    FALSE AS via_variant,
    alm.author_id,
    alm.display_name,
    alm.first,
    alm.first_initial,
    alm.middle,
    alm.last,
    alm.institution_ids,
    alm.topic_ids,
    alm.source_ids,
    alm.works_count,
    CAST(NULL AS STRING) AS v_pn_first,
    CAST(NULL AS STRING) AS v_pn_last,
    CAST(NULL AS STRING) AS v_kind
  FROM signal_reps e
  JOIN openalex.authors.authors_for_matching alm
    ON alm.block_key = e.block_key
    AND e.block_key != ''
),

-- 3b. Variant reach (oxjob #1341): any key the seat carries (primary + alternates) that a profile
--     carries (primary + alternates, capped at 10K-profile blocks in authors_for_matching_keys),
--     minus the pairs the primary block already produced. Compound surnames, reversed order and
--     transliteration variants land here. These candidates are held to a stricter bar (see eligible).
seat_keys_raw AS (
  SELECT tuple_key, block_key AS key, TRUE AS is_primary, pn_first AS s_first, pn_last AS s_last, pn_first AS p_first, pn_last AS p_last
  FROM signal_reps
  WHERE block_key IS NOT NULL AND block_key != ''
  UNION ALL
  SELECT tuple_key, p.key, FALSE, p.first AS s_first, p.last AS s_last, pn_first AS p_first, pn_last AS p_last
  FROM signal_reps
  LATERAL VIEW explode(COALESCE(alt_parses, ARRAY())) AS p
),

-- Reading kinds (oxjob #1341), inferred from how a reading's (first, last) relate to the side's primary parse:
--   primary; fold ('f:' keys); reversed (first/last swapped); reversed_middle (primary surname as given
--   name, a middle token as surname); surname_token (first kept, last is a token or join/split form of the primary surname);
--   middle (first kept, last is a middle token or middle + surname); other (romanizations, ALL-CAPS surname).
-- Opus 5.5 judged 935 variant matches of the 2026-09-24 batch: reversed_middle 15% precise, the pairs
-- surname_token|surname_token 31%, reversed|middle 35%, primary(seat)|middle(profile) 58%, middle|middle 76%;
-- everything else 81-95%. Dropping those lifts precision 72% -> 91% and keeps 58% of the cross-block pairs.
seat_keys AS (
  SELECT *,
    CASE WHEN is_primary THEN 'primary'
         WHEN key LIKE 'f:%' THEN 'fold'
         WHEN s_first = p_last AND s_last = p_first THEN 'reversed'
         WHEN s_first = p_last THEN 'reversed_middle'
         WHEN s_first = p_first AND (p_last = s_last OR array_contains(split(p_last, '[ -]'), s_last)
              OR regexp_replace(p_last, '[ -]', '') = regexp_replace(s_last, '[ -]', '') OR endswith(p_last, CONCAT(' ', s_last))) THEN 'surname_token'
         WHEN s_first = p_first THEN 'middle'
         ELSE 'other' END AS s_kind
  FROM seat_keys_raw
),

profile_keys AS (
  SELECT *,
    CASE WHEN is_primary THEN 'primary'
         WHEN key LIKE 'f:%' THEN 'fold'
         WHEN first = p_last AND last = p_first THEN 'reversed'
         WHEN first = p_last THEN 'reversed_middle'
         WHEN first = p_first AND (p_last = last OR array_contains(split(p_last, '[ -]'), last)
              OR regexp_replace(p_last, '[ -]', '') = regexp_replace(last, '[ -]', '') OR endswith(p_last, CONCAT(' ', last))) THEN 'surname_token'
         WHEN first = p_first THEN 'middle'
         ELSE 'other' END AS k_kind
  FROM openalex.authors.authors_for_matching_keys
),

-- one row per (seat tuple, profile): the first shared key by key order, with both sides' readings
-- under that key, so names are compared under the alternate reading (a reversed reading swaps
-- first/last; a surname-token reading uses the token as surname on both sides).
variant_ids AS (
  SELECT
    sk.tuple_key,
    fk.author_id,
    MIN_BY(sk.s_first, sk.key) AS v_pn_first,
    MIN_BY(sk.s_last,  sk.key) AS v_pn_last,
    MIN_BY(fk.first,   sk.key) AS v_cand_first,
    MIN_BY(fk.last,    sk.key) AS v_cand_last,
    MIN_BY(CONCAT(sk.s_kind, '|', fk.k_kind), sk.key) AS v_kind
  FROM seat_keys sk
  JOIN profile_keys fk
    ON fk.key = sk.key
    AND NOT (sk.is_primary AND fk.is_primary)
    -- allowed reading pairs only (see the kinds comment above)
    AND sk.s_kind <> 'reversed_middle' AND fk.k_kind <> 'reversed_middle'
    AND NOT (sk.s_kind = 'surname_token' AND fk.k_kind = 'surname_token')
    AND NOT (sk.s_kind = 'reversed' AND fk.k_kind = 'middle') AND NOT (sk.s_kind = 'middle' AND fk.k_kind = 'reversed')
    AND NOT (sk.s_kind = 'primary' AND fk.k_kind = 'middle')
    AND NOT (sk.s_kind = 'middle' AND fk.k_kind = 'middle')
  LEFT ANTI JOIN primary_candidates pc
    ON pc.tuple_key = sk.tuple_key AND pc.author_id = fk.author_id
  GROUP BY sk.tuple_key, fk.author_id
),

variant_candidates AS (
  SELECT
    vi.tuple_key,
    TRUE AS via_variant,
    alm.author_id,
    alm.display_name,
    vi.v_cand_first AS first,
    SUBSTRING(vi.v_cand_first, 1, 1) AS first_initial,
    CAST(NULL AS STRING) AS middle,
    vi.v_cand_last AS last,
    alm.institution_ids,
    alm.topic_ids,
    alm.source_ids,
    alm.works_count,
    vi.v_pn_first,
    vi.v_pn_last,
    vi.v_kind
  FROM variant_ids vi
  JOIN openalex.authors.authors_for_matching alm
    ON alm.author_id = vi.author_id
),

all_candidates AS (
  SELECT * FROM primary_candidates
  UNION ALL
  SELECT * FROM variant_candidates
),

blocked_candidates AS (
  SELECT 
    e.tuple_key,
    e.work_id,
    e.author_sequence,
    e.raw_author_name,
    e.pn_first,
    e.pn_first_initial,
    e.pn_middle,
    e.pn_last,
    e.block_key,
    e.institution_ids,
    e.topic_ids,
    e.work_source_ids,
    e.is_rematch,
    COALESCE(c.via_variant, FALSE) AS via_variant,
    c.v_kind,
    -- name-comparison columns (oxjob #1341): the seat's primary parse, or its alternate reading for a
    -- variant-reached candidate. pn_* stay the primary parse (they key the seat's group below).
    COALESCE(c.v_pn_first, e.pn_first) AS cmp_first,
    SUBSTRING(COALESCE(c.v_pn_first, e.pn_first), 1, 1) AS cmp_first_initial,
    CASE WHEN COALESCE(c.via_variant, FALSE) THEN NULL ELSE e.pn_middle END AS cmp_middle,
    COALESCE(c.v_pn_last, e.pn_last) AS cmp_last,
    c.author_id,
    c.display_name AS candidate_display_name,
    c.first AS cand_first,
    c.first_initial AS cand_first_initial,
    c.middle AS cand_middle,
    c.last AS cand_last,
    c.institution_ids as candidate_institution_ids,
    c.topic_ids as candidate_topic_ids,
    c.source_ids AS candidate_source_ids,
    c.works_count
  FROM signal_reps e
  LEFT JOIN all_candidates c
    ON c.tuple_key = e.tuple_key
),

with_match_signals AS (
  SELECT
    *,
    NAMED_STRUCT(
      'id', author_id,
      'display_name', candidate_display_name,
      'v_kind', v_kind,
      'has_topic', (size(topic_ids) > 0 AND size(candidate_topic_ids) > 0 AND arrays_overlap(candidate_topic_ids, topic_ids))
    ) AS candidate_obj,
    
    (size(institution_ids) > 0 AND size(candidate_institution_ids) > 0 
     AND arrays_overlap(candidate_institution_ids, institution_ids)) as has_inst,
    
    (size(topic_ids) > 0 AND size(candidate_topic_ids) > 0 
     AND arrays_overlap(candidate_topic_ids, topic_ids)) as has_topic,

     (SIZE(work_source_ids) > 0 AND SIZE(candidate_source_ids) > 0
     AND ARRAYS_OVERLAP(candidate_source_ids, work_source_ids)) AS has_source
  FROM blocked_candidates
),

-- oxjob #1341: a candidate reached only through an alternate key must also share an institution and
-- a source or topic with the seat, and hold at most 20 works; otherwise it is not a candidate at all.
-- Primary-block candidates are eligible unconditionally (unchanged behaviour). The works cap comes from
-- Opus 5.5 judging 742 variant matches of the 2026-09-24 batch: profiles <= 20 works 96.4% precise,
-- 21-100 works 84%, > 100 works ~50% (large common-name profiles are conflated clusters); a shared
-- topic did not help (reversed readings with a shared topic were 76%).
-- oxjob #1360: rematch seats (#649 worklist) never match through a variant key. Opus 5.5 on the 2026-09-26
-- nightly: variant matches of rematch seats 63% same person (52/82), of new works 89% (59/66).
with_eligibility AS (
  SELECT
    *,
    (author_id IS NOT NULL AND (NOT via_variant OR (NOT is_rematch AND has_inst AND (has_source OR has_topic) AND COALESCE(works_count, 0) <= 20))) AS eligible
  FROM with_match_signals
),

with_name_matches AS (
  SELECT
    *,
    -- 1: Exact Full Name (both have full first, full middle, same last)
    (LENGTH(cmp_first) > 1 AND LENGTH(cmp_middle) > 1 AND LENGTH(cand_first) > 1 AND LENGTH(cand_middle) > 1
     AND cmp_first = cand_first
     AND cmp_middle = cand_middle
     AND cmp_last = cand_last
    ) AND eligible as pattern_1_exact_full,

    -- 2: Exact First, Middle Initial match (batch has full first + middle initial only)
    (LENGTH(cmp_first) > 1 AND (cmp_middle IS NULL OR LENGTH(cmp_middle) <= 1)
     AND LENGTH(cand_first) > 1
     AND cmp_first = cand_first
     AND cmp_last = cand_last
     AND (cand_middle IS NULL OR cmp_middle IS NULL OR SUBSTRING(cmp_middle, 1, 1) = SUBSTRING(cand_middle, 1, 1))
    ) AND eligible as pattern_2_exact_first_mid_init,

    -- 3: Initials match to Full (batch has first initial + middle, candidate has full)
    (LENGTH(cmp_first) = 1 AND cmp_middle IS NOT NULL
     AND LENGTH(cand_first) > 1 AND cand_middle IS NOT NULL
     AND cmp_first_initial = cand_first_initial
     AND SUBSTRING(cmp_middle, 1, 1) = SUBSTRING(cand_middle, 1, 1)
     AND cmp_last = cand_last
    ) AND eligible as pattern_3_init_mid_init,

    -- 4: First Initial, Middle Initial match (both have only initials, no full names)
    (LENGTH(cmp_first) = 1 AND LENGTH(cand_first) = 1
     AND cmp_middle IS NOT NULL AND cand_middle IS NOT NULL
     AND LENGTH(cmp_middle) <= 1 AND LENGTH(cand_middle) <= 1
     AND cmp_first_initial = cand_first_initial
     AND SUBSTRING(cmp_middle, 1, 1) = SUBSTRING(cand_middle, 1, 1)
     AND cmp_last = cand_last
    ) AND eligible as pattern_4_first_init_mid_init,

    -- 5: Exact First, Exact Last (no middle)
    (LENGTH(cmp_first) > 1 AND LENGTH(cand_first) > 1
     AND cmp_first = cand_first
     AND cmp_last = cand_last
     AND cmp_middle IS NULL
    ) AND eligible as pattern_5_exact_first_last,

    -- 6: First Initial Only to Full (batch has first initial only, candidate has full first)
    (LENGTH(cmp_first) = 1 AND cmp_middle IS NULL
     AND LENGTH(cand_first) > 1
     AND cmp_first_initial = cand_first_initial
     AND cmp_last = cand_last
    ) AND eligible as pattern_6_first_init_to_full,

    -- 7: First Initial Only (both have only first initial, no middle)
    (LENGTH(cmp_first) = 1 AND LENGTH(cand_first) = 1
     AND cmp_middle IS NULL AND cand_middle IS NULL
     AND cmp_first_initial = cand_first_initial
     AND cmp_last = cand_last
    ) AND eligible as pattern_7_first_init_last,

    -- 8: Full Name to Initial (batch has full first, candidate has only initial)
    (LENGTH(cmp_first) > 1 AND LENGTH(cand_first) = 1
     AND cmp_first_initial = cand_first_initial
     AND cmp_last = cand_last
    ) AND eligible as pattern_8_full_to_init

  FROM with_eligibility
),

with_any_name_match AS (
  SELECT
    *,
    (pattern_1_exact_full OR pattern_2_exact_first_mid_init OR pattern_3_init_mid_init OR 
     pattern_4_first_init_mid_init OR pattern_5_exact_first_last OR pattern_6_first_init_to_full OR 
     pattern_7_first_init_last OR pattern_8_full_to_init) as any_name_match
  FROM with_name_matches
),

aggregated_counts AS (
  SELECT
    tuple_key,
    work_id,
    author_sequence,
    raw_author_name,
    block_key,
    institution_ids,
    pn_first,
    pn_first_initial,
    pn_middle,
    pn_last,
    work_source_ids,
    
    -- STRATEGY 1: Name Only (Unique)
    count_if(pattern_1_exact_full AND NOT via_variant) AS s1_n1, count_if(pattern_2_exact_first_mid_init AND NOT via_variant) AS s1_n2,
    count_if(pattern_3_init_mid_init AND NOT via_variant) AS s1_n3, count_if(pattern_4_first_init_mid_init AND NOT via_variant) AS s1_n4,
    count_if(pattern_5_exact_first_last AND NOT via_variant) AS s1_n5, count_if(pattern_6_first_init_to_full AND NOT via_variant) AS s1_n6,
    count_if(pattern_7_first_init_last AND NOT via_variant) AS s1_n7, count_if(pattern_8_full_to_init AND NOT via_variant) AS s1_n8,
    
    -- STRATEGY 2: Name + Institution
    count_if(pattern_1_exact_full AND has_inst AND NOT via_variant) AS s2_n1, count_if(pattern_2_exact_first_mid_init AND has_inst AND NOT via_variant) AS s2_n2,
    count_if(pattern_3_init_mid_init AND has_inst AND NOT via_variant) AS s2_n3, count_if(pattern_4_first_init_mid_init AND has_inst AND NOT via_variant) AS s2_n4,
    count_if(pattern_5_exact_first_last AND has_inst AND NOT via_variant) AS s2_n5, count_if(pattern_6_first_init_to_full AND has_inst AND NOT via_variant) AS s2_n6,

    -- STRATEGY 6: Name + Inst + Source
    count_if(pattern_1_exact_full AND has_inst AND has_source AND NOT via_variant) AS s6_n1,
    count_if(pattern_2_exact_first_mid_init AND has_inst AND has_source AND NOT via_variant) AS s6_n2,
    count_if(pattern_5_exact_first_last AND has_inst AND has_source AND NOT via_variant) AS s6_n5,
    count_if(pattern_6_first_init_to_full AND has_inst AND has_source AND NOT via_variant) AS s6_n6,
    count_if(pattern_7_first_init_last AND has_inst AND has_source AND NOT via_variant) AS s6_n7,

    -- STRATEGY 4: Name + Inst + Topic
    count_if(pattern_1_exact_full AND has_inst AND has_topic AND NOT via_variant) AS s4_n1,
    count_if(pattern_2_exact_first_mid_init AND has_inst AND has_topic AND NOT via_variant) AS s4_n2,
    count_if(pattern_5_exact_first_last AND has_inst AND has_topic AND NOT via_variant) AS s4_n5,
    count_if(pattern_6_first_init_to_full AND has_inst AND has_topic AND NOT via_variant) AS s4_n6,
    count_if(pattern_7_first_init_last AND has_inst AND has_topic AND NOT via_variant) AS s4_n7,

    -- STRATEGY 5: Name + Source
    count_if(pattern_1_exact_full AND has_source AND NOT via_variant) AS s5_n1,
    count_if(pattern_2_exact_first_mid_init AND has_source AND NOT via_variant) AS s5_n2,
    count_if(pattern_5_exact_first_last AND has_source AND NOT via_variant) AS s5_n5,
    count_if(pattern_6_first_init_to_full AND has_source AND NOT via_variant) AS s5_n6,
    count_if(pattern_7_first_init_last AND has_source AND NOT via_variant) AS s5_n7,

    -- STRATEGY 3: Name + Topic
    count_if(pattern_1_exact_full AND has_topic AND NOT via_variant) AS s3_n1,
    count_if(pattern_2_exact_first_mid_init AND has_topic AND NOT via_variant) AS s3_n2,
    count_if(pattern_5_exact_first_last AND has_topic AND NOT via_variant) AS s3_n5,
    
    -- CAPTURE MATCHED OBJECTS
    MAX(CASE WHEN pattern_1_exact_full AND NOT via_variant THEN candidate_obj END) AS match_s1_n1,
    MAX(CASE WHEN pattern_2_exact_first_mid_init AND NOT via_variant THEN candidate_obj END) AS match_s1_n2,
    MAX(CASE WHEN pattern_5_exact_first_last AND NOT via_variant THEN candidate_obj END) AS match_s1_n5,

    MAX(CASE WHEN pattern_1_exact_full AND has_inst AND NOT via_variant THEN candidate_obj END) AS match_s2_n1,
    MAX(CASE WHEN pattern_2_exact_first_mid_init AND has_inst AND NOT via_variant THEN candidate_obj END) AS match_s2_n2,
    MAX(CASE WHEN pattern_5_exact_first_last AND has_inst AND NOT via_variant THEN candidate_obj END) AS match_s2_n5,
    MAX(CASE WHEN pattern_6_first_init_to_full AND has_inst AND NOT via_variant THEN candidate_obj END) AS match_s2_n6,

    MAX(CASE WHEN pattern_1_exact_full AND has_inst AND has_source AND NOT via_variant THEN candidate_obj END) AS match_s6_n1,
    MAX(CASE WHEN pattern_2_exact_first_mid_init AND has_inst AND has_source AND NOT via_variant THEN candidate_obj END) AS match_s6_n2,
    MAX(CASE WHEN pattern_5_exact_first_last AND has_inst AND has_source AND NOT via_variant THEN candidate_obj END) AS match_s6_n5,
    MAX(CASE WHEN pattern_6_first_init_to_full AND has_inst AND has_source AND NOT via_variant THEN candidate_obj END) AS match_s6_n6,

    MAX(CASE WHEN pattern_1_exact_full AND has_source AND NOT via_variant THEN candidate_obj END) AS match_s5_n1,
    MAX(CASE WHEN pattern_2_exact_first_mid_init AND has_source AND NOT via_variant THEN candidate_obj END) AS match_s5_n2,
    MAX(CASE WHEN pattern_5_exact_first_last AND has_source AND NOT via_variant THEN candidate_obj END) AS match_s5_n5,
    MAX(CASE WHEN pattern_6_first_init_to_full AND has_source AND NOT via_variant THEN candidate_obj END) AS match_s5_n6,
    
    MAX(CASE WHEN pattern_1_exact_full AND has_inst AND has_topic AND NOT via_variant THEN candidate_obj END) AS match_s4_n1,
    MAX(CASE WHEN pattern_2_exact_first_mid_init AND has_inst AND has_topic AND NOT via_variant THEN candidate_obj END) AS match_s4_n2,
    MAX(CASE WHEN pattern_5_exact_first_last AND has_inst AND has_topic AND NOT via_variant THEN candidate_obj END) AS match_s4_n5,
    MAX(CASE WHEN pattern_6_first_init_to_full AND has_inst AND has_topic AND NOT via_variant THEN candidate_obj END) AS match_s4_n6,

    MAX(CASE WHEN pattern_1_exact_full AND has_topic AND NOT via_variant THEN candidate_obj END) AS match_s3_n1,
    MAX(CASE WHEN pattern_2_exact_first_mid_init AND has_topic AND NOT via_variant THEN candidate_obj END) AS match_s3_n2,
    MAX(CASE WHEN pattern_5_exact_first_last AND has_topic AND NOT via_variant THEN candidate_obj END) AS match_s3_n5,
    
    -- VARIANT TIERS (oxjob #1341): candidates reached only through an alternate key. Counted apart
    -- from the tiers above so they can never displace or tie a primary-block match; they fire only when
    -- the primary cascade found nothing. Full first name under the alternate reading, plus institution
    -- and a source (v6) or a topic (v4); eligibility already required inst + (source | topic).
    count_if(pattern_1_exact_full AND via_variant AND has_inst AND has_source) AS v6_n1,
    count_if(pattern_2_exact_first_mid_init AND via_variant AND has_inst AND has_source) AS v6_n2,
    count_if(pattern_5_exact_first_last AND via_variant AND has_inst AND has_source) AS v6_n5,
    count_if(pattern_1_exact_full AND via_variant AND has_inst AND has_topic) AS v4_n1,
    count_if(pattern_2_exact_first_mid_init AND via_variant AND has_inst AND has_topic) AS v4_n2,
    count_if(pattern_5_exact_first_last AND via_variant AND has_inst AND has_topic) AS v4_n5,
    MAX(CASE WHEN pattern_1_exact_full AND via_variant AND has_inst AND has_source THEN candidate_obj END) AS match_v6_n1,
    MAX(CASE WHEN pattern_2_exact_first_mid_init AND via_variant AND has_inst AND has_source THEN candidate_obj END) AS match_v6_n2,
    MAX(CASE WHEN pattern_5_exact_first_last AND via_variant AND has_inst AND has_source THEN candidate_obj END) AS match_v6_n5,
    MAX(CASE WHEN pattern_1_exact_full AND via_variant AND has_inst AND has_topic THEN candidate_obj END) AS match_v4_n1,
    MAX(CASE WHEN pattern_2_exact_first_mid_init AND via_variant AND has_inst AND has_topic THEN candidate_obj END) AS match_v4_n2,
    MAX(CASE WHEN pattern_5_exact_first_last AND via_variant AND has_inst AND has_topic THEN candidate_obj END) AS match_v4_n5,

    COUNT_IF(author_id IS NOT NULL AND NOT via_variant) AS total_candidates_in_block,   -- primary block only, as before
    COUNT_IF(any_name_match AND NOT via_variant) AS total_name_matches

  FROM with_any_name_match
  GROUP BY tuple_key, work_id, author_sequence, raw_author_name, block_key, institution_ids,
           pn_first, pn_first_initial, pn_middle, pn_last, work_source_ids
),

final_decision AS (
SELECT
  ac.tuple_key,
  ac.work_id,
  ac.author_sequence,
  ac.block_key,
  raw_author_name,
  institution_ids,
  pn_first,
  pn_first_initial,
  pn_last,
  work_source_ids,
  
  -- MATCH OUTCOME (all n8 tiers retired, oxjob #691: 11-63% judge-measured precision)
  CASE 
    WHEN om.orcid_author_id IS NOT NULL THEN 'MATCHED'
    WHEN (
      s1_n1=1 OR s1_n2=1 OR s1_n5=1 OR 
      s6_n1=1 OR s6_n2=1 OR s6_n5=1 OR s6_n6=1 OR
      s2_n1=1 OR s2_n2=1 OR s2_n5=1 OR s2_n6=1 OR
      s4_n1=1 OR s4_n2=1 OR s4_n5=1 OR s4_n6=1 OR
      s5_n1=1 OR s5_n2=1 OR s5_n5=1 OR s5_n6=1 OR
      s3_n1=1 OR s3_n2=1 OR s3_n5=1 OR
      v6_n1=1 OR v6_n2=1 OR v6_n5=1
    ) THEN 'MATCHED'
    WHEN total_candidates_in_block = 0 THEN 'NO_CANDIDATES'
    ELSE 'AMBIGUOUS'
  END AS match_outcome,

  -- NAME-BASED AUTHOR ID (existing name-pattern cascade)
  CASE 
    WHEN s1_n1 = 1 THEN match_s1_n1.id
    WHEN s1_n2 = 1 THEN match_s1_n2.id
    WHEN s1_n5 = 1 THEN match_s1_n5.id
    
    WHEN s6_n1 = 1 THEN match_s6_n1.id
    WHEN s6_n2 = 1 THEN match_s6_n2.id
    WHEN s6_n5 = 1 THEN match_s6_n5.id
    WHEN s6_n6 = 1 THEN match_s6_n6.id

    WHEN s2_n1 = 1 THEN match_s2_n1.id
    WHEN s2_n2 = 1 THEN match_s2_n2.id
    WHEN s2_n5 = 1 THEN match_s2_n5.id
    WHEN s2_n6 = 1 THEN match_s2_n6.id

    WHEN s4_n1 = 1 THEN match_s4_n1.id
    WHEN s4_n2 = 1 THEN match_s4_n2.id
    WHEN s4_n5 = 1 THEN match_s4_n5.id
    WHEN s4_n6 = 1 THEN match_s4_n6.id

    WHEN s5_n1 = 1 THEN match_s5_n1.id
    WHEN s5_n2 = 1 THEN match_s5_n2.id
    WHEN s5_n5 = 1 THEN match_s5_n5.id
    WHEN s5_n6 = 1 THEN match_s5_n6.id

    WHEN s3_n1 = 1 THEN match_s3_n1.id
    WHEN s3_n2 = 1 THEN match_s3_n2.id
    WHEN s3_n5 = 1 THEN match_s3_n5.id

    -- oxjob #1341: variant tiers, only when nothing above fired. v4 (inst + topic) is counted but not
    -- used: Opus 5.5 judged 8 of 14 such matches right on the 2026-09-23 batch (reversed common pinyin
    -- names with a shared topic), v6 (inst + source) 42 of 45.
    WHEN v6_n1 = 1 THEN match_v6_n1.id
    WHEN v6_n2 = 1 THEN match_v6_n2.id
    WHEN v6_n5 = 1 THEN match_v6_n5.id

    ELSE NULL
  END AS name_author_id,

  -- WHICH name tier fired (observational, oxjob #640): same WHEN order as
  -- name_author_id above, so this names the tier that produced that id.
  -- Consumed by AuthorshipDailyMetrics; no matching logic reads it.
  CASE 
    WHEN s1_n1 = 1 THEN 's1_n1'
    WHEN s1_n2 = 1 THEN 's1_n2'
    WHEN s1_n5 = 1 THEN 's1_n5'

    WHEN s6_n1 = 1 THEN 's6_n1'
    WHEN s6_n2 = 1 THEN 's6_n2'
    WHEN s6_n5 = 1 THEN 's6_n5'
    WHEN s6_n6 = 1 THEN 's6_n6'

    WHEN s2_n1 = 1 THEN 's2_n1'
    WHEN s2_n2 = 1 THEN 's2_n2'
    WHEN s2_n5 = 1 THEN 's2_n5'
    WHEN s2_n6 = 1 THEN 's2_n6'

    WHEN s4_n1 = 1 THEN 's4_n1'
    WHEN s4_n2 = 1 THEN 's4_n2'
    WHEN s4_n5 = 1 THEN 's4_n5'
    WHEN s4_n6 = 1 THEN 's4_n6'

    WHEN s5_n1 = 1 THEN 's5_n1'
    WHEN s5_n2 = 1 THEN 's5_n2'
    WHEN s5_n5 = 1 THEN 's5_n5'
    WHEN s5_n6 = 1 THEN 's5_n6'

    WHEN s3_n1 = 1 THEN 's3_n1'
    WHEN s3_n2 = 1 THEN 's3_n2'
    WHEN s3_n5 = 1 THEN 's3_n5'

    WHEN v6_n1 = 1 THEN 'v6_n1'
    WHEN v6_n2 = 1 THEN 'v6_n2'
    WHEN v6_n5 = 1 THEN 'v6_n5'
    ELSE NULL
  END AS name_match_tier,

  -- oxjob #1341: reading pair and topic overlap of the chosen variant match (observability)
  CASE WHEN v6_n1 = 1 THEN match_v6_n1.v_kind WHEN v6_n2 = 1 THEN match_v6_n2.v_kind WHEN v6_n5 = 1 THEN match_v6_n5.v_kind END AS via_kind,
  CASE WHEN v6_n1 = 1 THEN match_v6_n1.has_topic WHEN v6_n2 = 1 THEN match_v6_n2.has_topic WHEN v6_n5 = 1 THEN match_v6_n5.has_topic END AS via_has_topic,

  -- ORCID tier (global, most-cited holder). orcid_match_count = GLOBAL number
  -- of profiles holding the ORCID (0 = no holder or no usable ORCID).
  COALESCE(om.orcid_match_count, 0) AS orcid_match_count,
  om.orcid_author_id

FROM aggregated_counts ac
LEFT JOIN orcid_matches om
  ON ac.work_id = om.work_id
  AND ac.author_sequence = om.author_sequence
)
SELECT
  ap.work_id,
  ap.author_sequence,
  fd.block_key,
  ap.raw_author_name,
  fd.institution_ids,
  fd.pn_first,
  fd.pn_first_initial,
  fd.pn_last,
  fd.work_source_ids,
  fd.match_outcome,
  fd.name_author_id,
  fd.name_match_tier,
  -- oxjob #1341: TRUE when the final assignment came from a variant tier (alternate block key)
  (fd.orcid_author_id IS NULL AND fd.name_author_id IS NOT NULL AND fd.name_match_tier LIKE 'v%') AS via_variant,
  fd.via_kind,
  fd.via_has_topic,
  fd.orcid_match_count,
  fd.orcid_author_id,
  -- FINAL AUTHOR ID: ORCID wins over the name cascade
  COALESCE(fd.orcid_author_id, fd.name_author_id) AS existing_author_id,
  CASE 
    WHEN fd.orcid_author_id IS NOT NULL THEN 'orcid'
    WHEN fd.name_author_id IS NOT NULL THEN 'name'
    ELSE NULL
  END AS match_method,
  -- QA: ORCID picked a different author than the name cascade would have
  (fd.orcid_author_id IS NOT NULL AND fd.name_author_id IS NOT NULL
   AND fd.orcid_author_id <> fd.name_author_id) AS orcid_name_conflict,
  -- QA: ORCID match with zero name corroboration (name cascade found nothing).
  -- This is the population to sample for wrong-ORCID publisher metadata.
  (fd.orcid_author_id IS NOT NULL AND fd.name_author_id IS NULL) AS orcid_blind_match
FROM authors_prepared ap
JOIN final_decision fd
  ON fd.tuple_key = ap.tuple_key AND fd.raw_author_name = ap.raw_author_name;

### Step 3: Cluster Unmatched & Mint New IDs

In [ ]:
-- A. Get the current High Water Mark
DECLARE OR REPLACE VARIABLE max_id BIGINT;
-- High-water mark includes deleted profiles (openalex.authors.deleted_authors, oxjob #1354):
-- deleting the newest empty profiles must never let their ids be re-issued.
SET VARIABLE max_id = (
  SELECT GREATEST(
    (SELECT MAX(id) FROM openalex.authors.authors),
    (SELECT COALESCE(MAX(author_id), 0) FROM openalex.authors.deleted_authors)
  )
);

-- B. Seat -> cluster (oxjob #444). Persisted so Step 5 and the mint-provenance metric read the same
--    key instead of recomputing it. A seat carrying a USABLE ORCID clusters on the ORCID (one person,
--    two works in one batch, different affiliations used to mint two same-ORCID profiles); everything
--    else clusters on name + (institutions | sources) as before.
CREATE OR REPLACE TABLE openalex.authors.author_matching_mint_seats AS
WITH orcid_use AS (
    -- same guard as the ORCID tier: an ORCID on >1 authorship of one work is publisher-stamped
    SELECT work_id, author_sequence, raw_orcid,
           CASE WHEN COUNT(*) OVER (PARTITION BY work_id, raw_orcid) = 1 THEN raw_orcid END AS usable_orcid
    FROM (SELECT work_id, author_sequence, NULLIF(TRIM(authorship_struct.raw_orcid), '') AS raw_orcid
          FROM openalex.authors.author_matching_batch)
    WHERE raw_orcid IS NOT NULL
)
SELECT
    pa.work_id,
    pa.author_sequence,
    pa.raw_author_name,
    pa.match_outcome,
    ou.raw_orcid,
    ou.usable_orcid,
    CASE
        WHEN ou.usable_orcid IS NOT NULL THEN xxhash64('orcid', ou.usable_orcid)
        ELSE xxhash64(
            -- 1. NAME PART: Normalized if available, else Raw
            CASE
                WHEN pa.pn_first IS NOT NULL AND pa.pn_first != '' AND pa.pn_last IS NOT NULL
                THEN CONCAT(pa.pn_first, ' ', pa.pn_last)
                WHEN pa.pn_first_initial IS NOT NULL AND pa.pn_first_initial != '' AND pa.pn_last IS NOT NULL
                THEN CONCAT(pa.pn_first_initial, ' ', pa.pn_last)
                ELSE LOWER(TRIM(pa.raw_author_name))
            END,
            -- 2. SIGNAL PART: Institutions -> Sources
            CASE
                WHEN SIZE(b.all_institution_ids) > 0
                THEN concat_ws('|', sort_array(b.all_institution_ids))
                ELSE concat_ws('|', sort_array(pa.work_source_ids))
            END
        )
    END AS cluster_hash
FROM openalex.authors.pending_author_assignments pa
--  Join Batch to get 'all_institution_ids'
LEFT JOIN openalex.authors.author_matching_batch b
    ON pa.work_id = b.work_id AND pa.author_sequence = b.author_sequence
LEFT JOIN orcid_use ou
    ON pa.work_id = ou.work_id AND pa.author_sequence = ou.author_sequence
LEFT JOIN openalex.works.work_authors existing
    ON pa.work_id = existing.work_id
    AND pa.author_sequence = existing.author_sequence
WHERE
    -- Only unmatched records
    pa.match_outcome <> 'MATCHED'
    -- ensure we haven't already assigned an ID in a previous run
    AND existing.author_id IS NULL
    -- Safety: Ensure we actually have a name string to hash
    AND pa.raw_author_name IS NOT NULL
    AND TRIM(pa.raw_author_name) <> '';

-- C. Cluster and Mint
CREATE OR REPLACE TABLE openalex.authors.author_matching_new_author_queue AS
WITH unique_clusters AS (
    SELECT
        cluster_hash,
        MAX_BY(raw_author_name, length(raw_author_name)) as raw_display_name,
        -- guarded: the publisher-stamped copy of a coauthor's ORCID never lands on a new profile (#444)
        MAX(usable_orcid) as orcid,
        monotonically_increasing_id() as batch_row_id
    FROM openalex.authors.author_matching_mint_seats
    GROUP BY cluster_hash
)
SELECT
    uc.cluster_hash,
    CASE
        WHEN SIZE(SPLIT(uc.raw_display_name, ',')) = 2 THEN
            TRIM(SPLIT(uc.raw_display_name, ',')[1]) || ' ' || TRIM(SPLIT(uc.raw_display_name, ',')[0])
        ELSE
            uc.raw_display_name
    END AS display_name,
    uc.orcid,
    max_id + ROW_NUMBER() OVER (ORDER BY uc.batch_row_id) AS new_author_id
FROM unique_clusters uc;

In [0]:
-- review: today's match rates (display only; the rows above are the durable copy)
SELECT 
    pa.match_outcome, 
    COUNT(*) as count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as percentage
FROM openalex.authors.pending_author_assignments pa
GROUP BY pa.match_outcome

UNION ALL

-- 2. MINTING STATS
SELECT 
    'NEW_AUTHORS_TO_CREATE' as match_outcome,
    COUNT(*) as count,
    NULL as percentage
FROM openalex.authors.author_matching_new_author_queue;

### Step 4: Write New Authors to Profiles

In [ ]:
INSERT INTO openalex.authors.authors 
    (id, display_name, full_name, orcid, created_date, updated_date)
SELECT 
    new_author_id AS id,
    display_name,
    display_name AS full_name,
    orcid,
    current_timestamp() AS created_date,
    current_timestamp() AS updated_date
FROM openalex.authors.author_matching_new_author_queue;

### Step 5: Consolidate Decisions

In [ ]:
CREATE OR REPLACE TEMPORARY VIEW batch_author_decisions AS
SELECT
    b.work_id,
    b.author_sequence,
    b.raw_author_name,

    COALESCE(pa.existing_author_id, q.new_author_id) AS final_author_id

FROM openalex.authors.author_matching_batch b

LEFT JOIN openalex.authors.pending_author_assignments pa
    ON b.work_id = pa.work_id
    AND b.author_sequence = pa.author_sequence

-- seat -> cluster key persisted in Step 3 (only unmatched seats are in it), oxjob #444
LEFT JOIN openalex.authors.author_matching_mint_seats ms
    ON b.work_id = ms.work_id
    AND b.author_sequence = ms.author_sequence

LEFT JOIN openalex.authors.author_matching_new_author_queue q
    ON q.cluster_hash = ms.cluster_hash;

### Step 6: Update work_authors — author_id only

In [0]:
MERGE INTO openalex.works.work_authors AS target
USING batch_author_decisions AS source
ON target.work_id = source.work_id
   AND target.author_sequence = source.author_sequence

WHEN MATCHED THEN
    UPDATE SET
        target.author_id = source.final_author_id,
        target.raw_author_name = source.raw_author_name,
        target.updated_at = current_timestamp()

WHEN NOT MATCHED THEN
    INSERT (work_id, author_sequence, author_id, raw_author_name, created_at, updated_at)
    VALUES (source.work_id, source.author_sequence, source.final_author_id,
            source.raw_author_name, current_timestamp(), current_timestamp())

### Monitoring: self-reported run metrics (oxjob #1116)

The job that does the work writes its own metrics. Plain aggregates of the run-state tables this notebook built go to the shared tall table `openalex.monitoring.metrics` under `component = 'author_matching'`, `source = 'MatchAuthors'`, delete-then-insert per (date, component, source). Only metrics a check in `monitoring/checks/author_matching.yaml` reads are emitted. State and sampled-quality metrics stay in the observer (`notebooks/metrics/AuthorshipDailyMetrics`).

In [ ]:
CREATE SCHEMA IF NOT EXISTS openalex.monitoring

In [ ]:
CREATE TABLE IF NOT EXISTS openalex.monitoring.metrics (
  snapshot_date DATE NOT NULL,
  component     STRING NOT NULL,
  metric        STRING NOT NULL,
  dimension     STRING,
  value         DOUBLE NOT NULL,
  source        STRING NOT NULL,
  computed_at   TIMESTAMP NOT NULL
) USING DELTA
CLUSTER BY (component, snapshot_date)

In [ ]:
DELETE FROM openalex.monitoring.metrics
WHERE snapshot_date = current_date() AND component = 'author_matching' AND source = 'MatchAuthors'

In [ ]:
-- Match outcomes, tiers, ORCID sanity, batch size, mints: plain aggregates of the run-state tables.
INSERT INTO openalex.monitoring.metrics (snapshot_date, component, metric, dimension, value, source, computed_at)
SELECT current_date(), 'author_matching', metric, dimension, CAST(value AS DOUBLE), 'MatchAuthors', current_timestamp()
FROM (
  SELECT 'match_outcome' AS metric, match_outcome AS dimension, COUNT(*) AS value
  FROM openalex.authors.pending_author_assignments GROUP BY match_outcome
  UNION ALL
  SELECT 'match_tier',
         CASE WHEN match_method = 'orcid' THEN 'orcid' ELSE COALESCE(name_match_tier, '(none)') END,
         COUNT(*)
  FROM openalex.authors.pending_author_assignments
  GROUP BY CASE WHEN match_method = 'orcid' THEN 'orcid' ELSE COALESCE(name_match_tier, '(none)') END
  UNION ALL
  SELECT 'orcid_qa', 'name_conflict', SUM(CASE WHEN orcid_name_conflict THEN 1 ELSE 0 END)
  FROM openalex.authors.pending_author_assignments
  UNION ALL
  SELECT 'orcid_qa', 'blind_match', SUM(CASE WHEN orcid_blind_match THEN 1 ELSE 0 END)
  FROM openalex.authors.pending_author_assignments
  UNION ALL
  SELECT 'orcid_qa', 'splinter_orcid', SUM(CASE WHEN orcid_match_count > 1 THEN 1 ELSE 0 END)
  FROM openalex.authors.pending_author_assignments
  UNION ALL
  SELECT 'batch_seats', NULL, COUNT(*) FROM openalex.authors.pending_author_assignments
  UNION ALL
  SELECT 'new_authors_minted', NULL, COUNT(*) FROM openalex.authors.author_matching_new_author_queue
  UNION ALL
  SELECT 'new_authors_minted', 'with_orcid', SUM(CASE WHEN orcid IS NOT NULL THEN 1 ELSE 0 END)
  FROM openalex.authors.author_matching_new_author_queue
  UNION ALL
  -- oxjob #444: a mint whose ORCID an OLDER profile already holds means the ORCID tier's holder
  -- lookup missed it (stale/wrong lookup: 2026-07-15/16 minted 59K in two nights). Expect ~0.
  SELECT 'new_authors_minted', 'dup_of_existing_holder', COUNT(DISTINCT q.new_author_id)
  FROM openalex.authors.author_matching_new_author_queue q
  JOIN openalex.authors.authors a ON a.orcid = q.orcid
  CROSS JOIN (SELECT MIN(new_author_id) AS min_id FROM openalex.authors.author_matching_new_author_queue) f
  WHERE q.orcid IS NOT NULL AND a.id < f.min_id
  UNION ALL
  -- mint-path seats whose ORCID was publisher-stamped on >1 authorship of the work (not usable)
  SELECT 'orcid_qa', 'mint_seats_stamped_orcid', SUM(CASE WHEN raw_orcid IS NOT NULL AND usable_orcid IS NULL THEN 1 ELSE 0 END)
  FROM openalex.authors.author_matching_mint_seats
)
WHERE value IS NOT NULL

In [ ]:
-- Mint provenance: a cluster minted although an AMBIGUOUS feeder seat existed is the
-- splinter-risk pool; NO_CANDIDATES mints are clean-new. Reads the persisted seat -> cluster
-- table from Step 3 (oxjob #444), so it cannot drift from the key the mint used.
INSERT INTO openalex.monitoring.metrics (snapshot_date, component, metric, dimension, value, source, computed_at)
WITH cluster_prov AS (
  SELECT q.cluster_hash,
         MAX(CASE WHEN ms.match_outcome = 'AMBIGUOUS' THEN 1 ELSE 0 END) AS any_ambiguous
  FROM openalex.authors.author_matching_new_author_queue q
  JOIN openalex.authors.author_matching_mint_seats ms ON q.cluster_hash = ms.cluster_hash
  GROUP BY q.cluster_hash
)
SELECT current_date(), 'author_matching', 'new_authors_minted',
       CASE WHEN any_ambiguous = 1 THEN 'from_ambiguous' ELSE 'from_no_candidates' END,
       CAST(COUNT(*) AS DOUBLE), 'MatchAuthors', current_timestamp()
FROM cluster_prov
GROUP BY CASE WHEN any_ambiguous = 1 THEN 'from_ambiguous' ELSE 'from_no_candidates' END

### Assignment log — durable per-seat copy of this run's decisions

One row per seat decided tonight (outcome, tier, author, block size) with the attribution the run-state table lacks: primary source, provenance, origin (new work / rematch / restamped). Delete-then-insert per `run_date`, 120-day retention. The judge, the impossible-name check and the source/skew/wave metrics all read this table.

In [ ]:
CREATE TABLE IF NOT EXISTS openalex.authors.author_assignment_log (
  run_date            DATE,
  work_id             BIGINT,
  author_sequence     INT,
  raw_author_name     STRING,
  block_key           STRING,
  block_size          INT,
  match_outcome       STRING,
  match_method        STRING,
  name_match_tier     STRING,
  existing_author_id  BIGINT,
  orcid_author_id     BIGINT,
  orcid_match_count   INT,
  institution_ids     ARRAY<STRING>,
  work_source_ids     ARRAY<STRING>,
  primary_source_id   STRING,
  primary_source_name STRING,
  provenance          STRING,
  work_created_date   DATE,
  origin              STRING,
  logged_at           TIMESTAMP
) PARTITIONED BY (run_date)

In [ ]:
DELETE FROM openalex.authors.author_assignment_log WHERE run_date = current_date()

In [ ]:
-- origin: 'rematch' = admitted by the #649 worklist; 'new_work' = work created for this run;
-- 'restamped' = an older work whose updated_date bump pulled its unbound seats back in.
INSERT INTO openalex.authors.author_assignment_log
  (run_date, work_id, author_sequence, raw_author_name, block_key, block_size, match_outcome, match_method,
   name_match_tier, existing_author_id, orcid_author_id, orcid_match_count, institution_ids, work_source_ids,
   primary_source_id, primary_source_name, provenance, work_created_date, origin, logged_at, via_variant)
WITH block_sizes AS (
    SELECT block_key, COUNT(*) AS block_size
    FROM openalex.authors.authors_for_matching
    WHERE block_key IN (SELECT DISTINCT block_key FROM openalex.authors.pending_author_assignments WHERE block_key IS NOT NULL)
    GROUP BY block_key
),
worklist AS (
    SELECT DISTINCT work_id FROM openalex.authors.author_rematch_applied WHERE run_date = current_date()
)
SELECT current_date(),
       p.work_id,
       CAST(p.author_sequence AS INT),
       p.raw_author_name,
       p.block_key,
       CAST(COALESCE(b.block_size, 0) AS INT),
       p.match_outcome,
       p.match_method,
       p.name_match_tier,
       p.existing_author_id,
       p.orcid_author_id,
       CAST(p.orcid_match_count AS INT),
       CAST(mb.all_institution_ids AS ARRAY<STRING>),
       CAST(p.work_source_ids AS ARRAY<STRING>),
       w.primary_location.source.id,
       w.primary_location.source.display_name,
       w.primary_location.provenance,
       w.created_date,
       CASE WHEN wl.work_id IS NOT NULL THEN 'rematch'
            WHEN w.created_date >= current_date() - INTERVAL 1 DAY THEN 'new_work'
            ELSE 'restamped' END,
       current_timestamp(),
       COALESCE(p.via_variant, FALSE)
FROM openalex.authors.pending_author_assignments p
LEFT JOIN block_sizes b ON p.block_key = b.block_key
LEFT JOIN openalex.authors.author_matching_batch mb ON p.work_id = mb.work_id AND p.author_sequence = mb.author_sequence
LEFT JOIN openalex.works.openalex_works_base w ON p.work_id = w.id
LEFT JOIN worklist wl ON p.work_id = wl.work_id

In [ ]:
DELETE FROM openalex.authors.author_assignment_log WHERE run_date < current_date() - INTERVAL 120 DAYS

### Monitoring: metrics derived from the assignment log

Source mix (top 5), block-join skew, the largest single-author gain, the name-concentration wave detector (named), the impossible-name check, and the author count. Same table and source as the run outcomes above.

In [ ]:
INSERT INTO openalex.monitoring.metrics (snapshot_date, component, metric, dimension, value, source, computed_at)
SELECT current_date(), 'author_matching', metric, dimension, CAST(value AS DOUBLE), 'MatchAuthors', current_timestamp()
FROM (
  SELECT 'assignment_log_rows' AS metric, CAST(NULL AS STRING) AS dimension, COUNT(*) AS value
  FROM openalex.authors.author_assignment_log WHERE run_date = current_date()
  UNION ALL
  -- top 5 sources by seats in tonight's batch: a source flooding the batch shows up here by name
  SELECT 'seats_by_source', d, seats FROM (
    SELECT CONCAT(regexp_replace(primary_source_id, '.*/', ''), ' · ', LEFT(COALESCE(primary_source_name, ''), 80)) AS d,
           COUNT(*) AS seats
    FROM openalex.authors.author_assignment_log
    WHERE run_date = current_date() AND primary_source_id IS NOT NULL
    GROUP BY 1 ORDER BY seats DESC LIMIT 5)
)
WHERE value IS NOT NULL

In [ ]:
-- Most seats bound to a single existing author tonight (top 5, named). Name-agnostic absorber
-- detector: a shared ORCID or an org pseudo-author absorbing a roster shows here even when the
-- names agree; a hyperauthorship paper adds one seat per author, not thousands to one.
INSERT INTO openalex.monitoring.metrics (snapshot_date, component, metric, dimension, value, source, computed_at)
SELECT current_date(), 'author_matching', 'author_gain_top',
       CONCAT('A', l.existing_author_id, ' · ', LEFT(COALESCE(oa.display_name, ''), 60)),
       CAST(COUNT(*) AS DOUBLE), 'MatchAuthors', current_timestamp()
FROM openalex.authors.author_assignment_log l
LEFT JOIN openalex.authors.openalex_authors oa ON oa.id = l.existing_author_id
WHERE l.run_date = current_date() AND l.match_outcome = 'MATCHED' AND l.existing_author_id IS NOT NULL
GROUP BY l.existing_author_id, oa.display_name
ORDER BY COUNT(*) DESC LIMIT 5

In [ ]:
-- Block-join skew: the candidate join costs SUM(block_size) over the DISTINCT signal tuples
-- MatchAuthors dedupes to (name, institutions, sources); wall time is set by the largest block.
CREATE OR REPLACE TEMPORARY VIEW assign_tuples AS
SELECT block_key, MAX(block_size) AS block_size, COUNT(*) AS seats
FROM openalex.authors.author_assignment_log
WHERE run_date = current_date() AND block_key IS NOT NULL AND block_key <> ''
GROUP BY block_key, raw_author_name, institution_ids, work_source_ids

In [ ]:
INSERT INTO openalex.monitoring.metrics (snapshot_date, component, metric, dimension, value, source, computed_at)
SELECT current_date(), 'author_matching', metric, dimension, CAST(value AS DOUBLE), 'MatchAuthors', current_timestamp()
FROM (
  WITH per_block AS (
    SELECT block_key, SUM(block_size) AS tuple_rows, SUM(seats * block_size) AS seat_rows,
           SUM(CASE WHEN block_size > 100000 THEN seats ELSE 0 END) AS seats_gt100k
    FROM assign_tuples GROUP BY block_key
  )
  SELECT 'block_skew' AS metric, 'join_rows_tuples' AS dimension, SUM(tuple_rows) AS value FROM per_block
  UNION ALL SELECT 'block_skew', 'join_rows_seats', SUM(seat_rows) FROM per_block
  UNION ALL SELECT 'block_skew', 'max_block_rows_tuples', MAX(tuple_rows) FROM per_block
  UNION ALL SELECT 'block_skew', 'max_block_rows_seats', MAX(seat_rows) FROM per_block
  UNION ALL SELECT 'block_skew', 'seats_in_blocks_gt100k', SUM(seats_gt100k) FROM per_block
)
WHERE value IS NOT NULL

In [ ]:
INSERT INTO openalex.monitoring.metrics (snapshot_date, component, metric, dimension, value, source, computed_at)
SELECT current_date(), 'author_matching', metric, dimension, CAST(value AS DOUBLE), 'MatchAuthors', current_timestamp()
FROM (
  -- Name-concentration wave detector: one raw_author_name flooding the batch = an org
  -- pseudo-author wave or a parser bug. Surfaced BY NAME (caught Geoscience Australia 2026-07-30).
  SELECT 'name_concentration_top' AS metric, LEFT(raw_author_name, 120) AS dimension, COUNT(*) AS value
  FROM openalex.authors.pending_author_assignments
  WHERE raw_author_name IS NOT NULL AND TRIM(raw_author_name) <> ''
  GROUP BY raw_author_name HAVING COUNT(*) >= 1000
  ORDER BY value DESC LIMIT 15
)
WHERE value IS NOT NULL

In [ ]:
-- Impossible-name check (Casey, 2026-09-14): every MATCHED seat vs the LIVE display name of the
-- profile it was bound to. #608 predicates: foreign_family = different surname family (~87% precision);
-- first_initial_clash = same surname, different first initial, seat's given name absent from the profile name.
-- ORCID matches have no other name gate.
CREATE OR REPLACE TEMPORARY VIEW assign_name_judged AS
WITH seats AS (
  SELECT l.match_method, l.raw_author_name AS base_name, l.existing_author_id AS author_id,
         COALESCE(NULLIF(TRIM(oa.display_name), ''), TRIM(oa.full_name)) AS bound_name
  FROM openalex.authors.author_assignment_log l
  JOIN openalex.authors.openalex_authors oa ON oa.id = l.existing_author_id
  WHERE l.run_date = current_date() AND l.match_outcome = 'MATCHED'
),
k AS (
  SELECT s.*, an_r.match_last AS r_last, an_r.match_first AS r_first,
         an_p.match_last AS p_last, an_p.match_first AS p_first,
         FILTER(SPLIT(LOWER(REGEXP_REPLACE(REGEXP_REPLACE(s.bound_name, '^([^,]+),(.*)$', '$2 $1'), '[.,\\-]+', ' ')), '\\s+'), t -> LENGTH(t) >= 2) AS p_tokens
  FROM seats s
  JOIN openalex.authors.author_names an_r ON TRIM(s.base_name) = an_r.raw_author_name
  JOIN openalex.authors.author_names an_p ON s.bound_name = an_p.raw_author_name
  WHERE an_r.match_last IS NOT NULL AND an_p.match_last IS NOT NULL
    AND s.base_name NOT RLIKE '(?i)^(n/?a|null|none|unknown|anonymous|et al\\.?|grd)\\b'
    AND NOT (s.base_name RLIKE '<[a-zA-Z/][^>]*>' OR s.base_name RLIKE '(?i)(vcard|begin:|;type=)')
    AND NOT (s.base_name RLIKE '[Ͱ-ϿЀ-ӿ؀-ۿᄀ-ᇿ぀-ヿ㄰-㆏㐀-䶿一-鿿가-힯豈-﫿]'
             OR s.bound_name RLIKE '[Ͱ-ϿЀ-ӿ؀-ۿᄀ-ᇿ぀-ヿ㄰-㆏㐀-䶿一-鿿가-힯豈-﫿]')
)
SELECT match_method, author_id, bound_name,
  CASE
    WHEN r_last <> p_last AND LENGTH(p_last) >= 2 AND LENGTH(r_last) >= 2
     AND INSTR(p_last, r_last) = 0 AND INSTR(r_last, p_last) = 0
     AND r_last <> COALESCE(p_first, '') AND COALESCE(r_first, '') <> p_last
     AND NOT (r_first IS NOT NULL AND r_first = p_first AND LENGTH(r_first) >= 3)
     AND SIZE(ARRAY_INTERSECT(
           FILTER(SLICE(SPLIT(TRIM(REGEXP_REPLACE(TRANSLATE(LOWER(REGEXP_REPLACE(REGEXP_REPLACE(base_name,'^([^,]+),(.*)$','$2 $1'),'[.,\\-]+',' ')),'áàäâãåéèëêíìïîóòöôõøúùüûñçýćčšžł','aaaaaaeeeeiiiioooooouuuuncyccszl'),'\\s+',' ')),' '),2,20), t -> LENGTH(t) >= 3),
           FILTER(SLICE(SPLIT(TRIM(REGEXP_REPLACE(TRANSLATE(LOWER(REGEXP_REPLACE(REGEXP_REPLACE(bound_name,'^([^,]+),(.*)$','$2 $1'),'[.,\\-]+',' ')),'áàäâãåéèëêíìïîóòöôõøúùüûñçýćčšžł','aaaaaaeeeeiiiioooooouuuuncyccszl'),'\\s+',' ')),' '),2,20), t -> LENGTH(t) >= 3)
         )) = 0
     AND levenshtein(r_last, p_last) > 1
     AND NOT (levenshtein(r_last, p_last) = 2 AND LEFT(COALESCE(r_first,''),1) = LEFT(COALESCE(p_first,''),1))
    THEN 'foreign_family'
    WHEN r_last = p_last AND r_first IS NOT NULL AND p_first IS NOT NULL
     AND LEFT(r_first, 1) <> LEFT(p_first, 1)
     AND LENGTH(r_first) >= 2 AND NOT ARRAY_CONTAINS(p_tokens, LOWER(r_first))
    THEN 'first_initial_clash'
  END AS cls
FROM k

In [ ]:
INSERT INTO openalex.monitoring.metrics (snapshot_date, component, metric, dimension, value, source, computed_at)
SELECT current_date(), 'author_matching', metric, dimension, CAST(value AS DOUBLE), 'MatchAuthors', current_timestamp()
FROM (
  SELECT 'assign_name_check' AS metric, CONCAT(cls, '|', match_method) AS dimension, COUNT(*) AS value
  FROM assign_name_judged WHERE cls IS NOT NULL GROUP BY 2
  UNION ALL
  SELECT 'assign_name_check_top', CONCAT('A', author_id, ' · ', LEFT(bound_name, 60)), c FROM (
    SELECT author_id, bound_name, COUNT(*) AS c FROM assign_name_judged WHERE cls = 'foreign_family'
    GROUP BY 1, 2 ORDER BY c DESC LIMIT 15)
)
WHERE value IS NOT NULL

In [ ]:
INSERT INTO openalex.monitoring.metrics (snapshot_date, component, metric, dimension, value, source, computed_at)
SELECT current_date(), 'author_matching', 'authors_total', NULL, CAST(COUNT(*) AS DOUBLE), 'MatchAuthors', current_timestamp()
FROM openalex.authors.authors

In [ ]:
SELECT metric, dimension, value
FROM openalex.monitoring.metrics
WHERE snapshot_date = current_date() AND component = 'author_matching' AND source = 'MatchAuthors'
ORDER BY metric, dimension